# SASV: AASIST + LFCC CM ensemble

Reuse saved score CSVs (no GPU):

```text
s_cm = β · s_cm_aasist + (1 − β) · s_cm_lfcc
s_sasv = s_asv + s_cm          # default (sum mode)
# or
s_sasv = α · s_asv + (1 − α) · s_cm   # optional grid
```

- AASIST scores: `runs/ecapa_plus_aasist_{split}/scores_*.csv`
- LFCC scores: `runs/ecapa_plus_lfcc_{split}/scores_*.csv`

Tune `β` (and optionally `α`) on **dev**; lock on **eval**.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "ensemble_fusion_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

from experiment_lib import DEFAULT_SASV, RUNS_DIR, ensure_sasv_on_path
from ensemble_fusion_lib import (
    eers_for_ensemble,
    load_aligned_cm_csvs,
    save_ensemble_run,
    sweep_alpha_beta,
    sweep_beta,
)

ensure_sasv_on_path(DEFAULT_SASV)
from metrics import get_all_EERs

OUT = RUNS_DIR / "ecapa_plus_aasist_lfcc_ens"
OUT.mkdir(parents=True, exist_ok=True)
print("OUT", OUT)

## Knobs

- `SWEEP_ALPHA = False` → only sweep β with `s_asv + s_cm` (recommended first)
- `SWEEP_ALPHA = True` → grid α×β (heavier; more overfit risk)

In [ ]:
SWEEP_ALPHA = False
N_BETA = 21
N_ALPHA = 11  # used only if SWEEP_ALPHA
RUN_EVAL = True
BETAS = np.linspace(0.0, 1.0, N_BETA)
ALPHAS = np.linspace(0.0, 1.0, N_ALPHA)

## 1. Load aligned **dev** scores

In [ ]:
DEV_A = RUNS_DIR / "ecapa_plus_aasist_dev" / "scores_dev.csv"
DEV_L = RUNS_DIR / "ecapa_plus_lfcc_dev" / "scores_dev.csv"
s_asv_d, s_aa_d, s_lf_d, keys_d = load_aligned_cm_csvs(DEV_A, DEV_L)
print(f"dev aligned trials: {len(keys_d)}")

# References
for name, preds in [
    ("aasist raw sum", (s_asv_d + s_aa_d).tolist()),
    ("lfcc raw sum", (s_asv_d + s_lf_d).tolist()),
    ("equal ens β=0.5 sum", (s_asv_d + 0.5 * s_aa_d + 0.5 * s_lf_d).tolist()),
]:
    sasv, sv, spf = get_all_EERs(preds, keys_d)
    print(f"dev {name:22s}  SASV={sasv*100:.4f}%  SV={sv*100:.4f}%  SPF={spf*100:.4f}%")

## 2. Tune on **dev**

In [ ]:
if SWEEP_ALPHA:
    best_dev, sweep_rows = sweep_alpha_beta(
        s_asv_d, s_aa_d, s_lf_d, keys_d,
        alphas=ALPHAS, betas=BETAS, sasv_root=DEFAULT_SASV,
    )
else:
    best_dev, sweep_rows = sweep_beta(
        s_asv_d, s_aa_d, s_lf_d, keys_d,
        betas=BETAS, mode="sum", sasv_root=DEFAULT_SASV,
    )

LOCKED_BETA = float(best_dev["beta"])
LOCKED_ALPHA = float(best_dev.get("alpha", 1.0))
LOCKED_MODE = best_dev.get("mode", "sum")
print("Locked:", {"mode": LOCKED_MODE, "alpha": LOCKED_ALPHA, "beta": LOCKED_BETA})
print(
    f"dev best  SASV={best_dev['sasv_eer_percent']:.4f}%  "
    f"SV={best_dev['sv_eer_percent']:.4f}%  SPF={best_dev['spf_eer_percent']:.4f}%"
)

if not SWEEP_ALPHA:
    print("\nβ sweep (sum mode):")
    for row in sweep_rows:
        mark = " <-- best" if abs(row["beta"] - LOCKED_BETA) < 1e-9 else ""
        print(
            f"  β={row['beta']:.2f}  SASV={row['sasv_eer_percent']:7.4f}%  "
            f"SV={row['sv_eer_percent']:7.4f}%  SPF={row['spf_eer_percent']:7.4f}%{mark}"
        )
else:
    print(f"\nGrid size: {len(sweep_rows)} (showing best only above)")

In [ ]:
dev_out = save_ensemble_run(
    split="dev",
    s_asv=s_asv_d,
    s_aasist=s_aa_d,
    s_lfcc=s_lf_d,
    keys=keys_d,
    metrics=best_dev,
    output_dir=OUT / "dev",
)
(OUT / "sweep_dev.json").write_text(
    json.dumps(
        {
            "locked": {
                "mode": LOCKED_MODE,
                "alpha": LOCKED_ALPHA,
                "beta": LOCKED_BETA,
            },
            "sweep": sweep_rows,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("Wrote", dev_out)

## 3. Locked **eval**

In [ ]:
if not RUN_EVAL:
    print("RUN_EVAL=False")
else:
    EVAL_A = RUNS_DIR / "ecapa_plus_aasist_eval" / "scores_eval.csv"
    EVAL_L = RUNS_DIR / "ecapa_plus_lfcc_eval" / "scores_eval.csv"
    s_asv_e, s_aa_e, s_lf_e, keys_e = load_aligned_cm_csvs(EVAL_A, EVAL_L)
    print(f"eval aligned trials: {len(keys_e)}")

    # References
    refs = {}
    for name, preds in [
        ("aasist_raw_sum", (s_asv_e + s_aa_e).tolist()),
        ("lfcc_raw_sum", (s_asv_e + s_lf_e).tolist()),
    ]:
        sasv, sv, spf = get_all_EERs(preds, keys_e)
        refs[name] = {
            "sasv_eer_percent": sasv * 100,
            "sv_eer_percent": sv * 100,
            "spf_eer_percent": spf * 100,
        }
        print(
            f"eval {name:16s}  SASV={sasv*100:.4f}%  SV={sv*100:.4f}%  SPF={spf*100:.4f}%"
        )

    eval_metrics = eers_for_ensemble(
        s_asv_e, s_aa_e, s_lf_e, keys_e,
        alpha=LOCKED_ALPHA, beta=LOCKED_BETA, mode=LOCKED_MODE,
        sasv_root=DEFAULT_SASV,
    )
    print(
        f"eval ensemble α={LOCKED_ALPHA:.2f} β={LOCKED_BETA:.2f} mode={LOCKED_MODE}  "
        f"SASV={eval_metrics['sasv_eer_percent']:.4f}%  "
        f"SV={eval_metrics['sv_eer_percent']:.4f}%  "
        f"SPF={eval_metrics['spf_eer_percent']:.4f}%"
    )

    eval_out = save_ensemble_run(
        split="eval",
        s_asv=s_asv_e,
        s_aasist=s_aa_e,
        s_lfcc=s_lf_e,
        keys=keys_e,
        metrics=eval_metrics,
        output_dir=OUT / "eval",
    )
    locked = {
        "locked_mode": LOCKED_MODE,
        "locked_alpha": LOCKED_ALPHA,
        "locked_beta": LOCKED_BETA,
        "tuned_on": "dev",
        "objective": "min SASV-EER",
        "dev_metrics": best_dev,
        "eval_metrics": eval_metrics,
        "eval_refs": refs,
    }
    (OUT / "locked_eval.json").write_text(json.dumps(locked, indent=2), encoding="utf-8")
    print("Wrote", eval_out)
    print(json.dumps(locked, indent=2))

## Done

Compare eval ensemble SASV-EER to AASIST-only raw sum (~1.14%).  
Outputs: `runs/ecapa_plus_aasist_lfcc_ens/`.